# Auditoria de Vazamento (Target Leakage)

**Objetivo:**  
Garantir que todas as variáveis da ABT (`gold/abt_base_prod`) respeitam o princípio **AS-OF** do Credit Scoring:
> cada feature deve ser calculada apenas com informações disponíveis **antes do início da safra** (momento de decisão).

Esta auditoria é pré-requisito para:
- Estudo de Público-Alvo
- Modelo Baseline de Behavior
- Definição de Política (Swap-in)


## Imports

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np
from urllib.parse import urlparse
from sklearn.metrics import roc_auc_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Helpers

In [2]:
def get_duckdb_connection(memory_limit: str = "9GB", threads: int = 4) -> duckdb.DuckDBPyConnection:
    endpoint = os.getenv("S3_ENDPOINT")
    if not endpoint:
        raise RuntimeError("S3_ENDPOINT nao definido")

    parsed = urlparse(endpoint)
    if not parsed.netloc:
        raise RuntimeError(f"S3_ENDPOINT invalido: {endpoint}")

    duckdb_endpoint = parsed.netloc
    con = duckdb.connect()

    con.execute(f"""
        INSTALL httpfs;
        LOAD httpfs;

        SET memory_limit='{memory_limit}';
        SET threads={threads};

        SET s3_endpoint='{duckdb_endpoint}';
        SET s3_access_key_id='{os.getenv("AWS_ACCESS_KEY_ID")}';
        SET s3_secret_access_key='{os.getenv("AWS_SECRET_ACCESS_KEY")}';
        SET s3_region='{os.getenv("AWS_DEFAULT_REGION", "us-east-1")}';
        SET s3_use_ssl=false;
        SET s3_url_style='path';
    """)
    return con


In [3]:
def feature_domain(col):
    return col.split("_")[0]

In [4]:
con = get_duckdb_connection()

ABT_PATH = "s3://lake/gold/abt_base_cmv/**/*.parquet"

df = con.execute(f"""
    SELECT *
    FROM read_parquet('{ABT_PATH}')
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
df.head()

,num_cpf,safra,prod,fpd,flag_instalacao,rec_qtd_l30d,rec_qtd_l60d,rec_qtd_l90d,rec_qtd_geral,rec_vlr_total_l30d,rec_vlr_total_l60d,rec_vlr_total_l90d,rec_vlr_total_geral,rec_vlr_avg_l30d,rec_vlr_avg_l60d,rec_vlr_avg_l90d,rec_vlr_avg_geral,rec_vlr_min_l30d,rec_vlr_min_l60d,rec_vlr_min_l90d,rec_vlr_min_geral,rec_vlr_max_l30d,rec_vlr_max_l60d,rec_vlr_max_l90d,rec_vlr_max_geral,rec_dat_primeira,rec_dat_ultima,rec_qtd_canais_distintos,rec_dias_desde_ultima,rec_dias_desde_primeira,rec_vlr_std_l30d,rec_vlr_std_l60d,rec_vlr_std_l90d,rec_vlr_std_geral,rec_vlr_coef_var_l30d,rec_vlr_coef_var_l60d,rec_vlr_coef_var_l90d,rec_ratio_qtd_l30d_l60d,rec_ratio_qtd_l60d_l90d,rec_ratio_vlr_l30d_l60d,rec_ratio_vlr_l60d_l90d,rec_flag_sem_recarga_l30d,rec_flag_sem_recarga_l60d,rec_flag_sem_recarga_l90d,pag_vlr_total_l30d,pag_vlr_total_l60d,pag_vlr_total_l90d,pag_vlr_total_geral,pag_vlr_avg_l30d,pag_vlr_avg_l60d,pag_vlr_avg_l90d,pag_vlr_avg_geral,pag_vlr_min_l30d,pag_vlr_min_l60d,pag_vlr_min_l90d,pag_vlr_min_geral,pag_vlr_max_l30d,pag_vlr_max_l60d,pag_vlr_max_l90d,pag_vlr_max_geral,pag_qtd_faturas_l30d,pag_qtd_faturas_l60d,pag_qtd_faturas_l90d,pag_qtd_faturas_geral,pag_qtd_vezes_com_juros,pag_dias_desde_ultimo_pagamento,pag_ticket_medio_l30d,pag_ticket_medio_l60d,pag_ticket_medio_l90d,pag_ticket_medio_geral,pag_share_faturas_com_juros_l30d,pag_share_faturas_com_juros_l60d,pag_share_faturas_com_juros_l90d,pag_share_faturas_com_juros_geral,pag_vlr_std_l30d,pag_vlr_std_l60d,pag_vlr_std_l90d,pag_flag_sem_pagamento_l30d,pag_flag_sem_pagamento_l60d,pag_flag_sem_pagamento_l90d,atr_vlr_max_l30d,atr_vlr_max_l60d,atr_vlr_max_l90d,atr_vlr_max_geral,atr_vlr_acumulado_l30d,atr_vlr_acumulado_l60d,atr_vlr_acumulado_l90d,atr_vlr_acumulado_geral,atr_qtd_faturas_atrasadas_l30d,atr_qtd_faturas_atrasadas_l60d,atr_qtd_faturas_atrasadas_l90d,atr_qtd_faturas_atrasadas_geral,atr_dat_ultima_ref,atr_dias_desde_ultimo_atraso,atr_ticket_medio_l30d,atr_ticket_medio_l60d,atr_ticket_medio_l90d,atr_ticket_medio_geral,atr_flag_recorrente_l30d,atr_flag_recorrente_l60d,...,cad_cep_3_digitos,cad_datadenascimento,cad_flag_mig2,cad_statusrf,cad_var_02,cad_var_03,cad_var_04,cad_var_05,cad_var_06,cad_var_07,cad_var_08,cad_var_09,cad_var_10,cad_var_11,cad_var_12,cad_var_13,cad_var_14,cad_var_15,cad_var_16,cad_var_17,cad_var_18,cad_var_19,cad_var_20,cad_var_21,cad_var_22,cad_var_23,cad_var_24,cad_var_25,tel_flag_mig2,tel_var_26,tel_var_27,tel_var_28,tel_var_29,tel_var_30,tel_var_31,tel_var_32,tel_var_33,tel_var_34,tel_var_35,tel_var_36,tel_var_37,tel_var_38,tel_var_39,tel_var_40,tel_var_41,tel_var_42,tel_var_43,tel_var_44,tel_var_45,tel_var_46,tel_var_47,tel_var_48,tel_var_49,tel_var_50,tel_var_51,tel_var_52,tel_var_53,tel_var_54,tel_var_55,tel_var_56,tel_var_57,tel_var_58,tel_var_59,tel_var_60,tel_var_61,tel_var_62,tel_var_63,tel_var_64,tel_var_65,tel_var_66,tel_var_67,tel_var_68,tel_var_69,tel_var_70,tel_var_71,tel_var_72,tel_var_73,tel_var_74,tel_var_75,tel_var_76,tel_var_77,tel_var_78,tel_var_79,tel_var_80,tel_var_81,tel_var_82,tel_var_83,tel_var_84,tel_var_85,tel_var_86,tel_var_87,tel_var_88,tel_var_89,tel_var_90,tel_var_91,tel_var_92,tel_var_93,run_id,ingestion_ts,ano_mes
0,X8YZN9UY7TZ,2024-10-01,CMV,False,True,9,17,24,61,80.0,160.0,215.0,555.0,8.888889,9.411765,8.958333,9.098361,0.0,0.0,0.0,0.0,20.0,20.0,20.0,20.0,2023-10-22,2024-09-28,3,3,345,8.936504,8.818013,8.594635,8.684670,1.005357,0.936914,0.959401,0.529412,0.708333,0.500000,0.744186,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,NaT,<NA>,NaN,NaN,NaN,NaN,0,0,...,244,1998-11-23,PRE,REGULAR,<NA>,40,0,1,<NA>,NaN,<NA>,9,None,NaN,NaT,NaT,<NA>,RJ,950,202311,None,AUX_EMRG,None,None,None,BOLSA_FAMILIA,None,AUX_EMRG BOLSA_FAMILIA,PRE,1,1,0.00,304.00,28.37,0.14,304.00,29.97,304.00,35.63,0.00,0.00,304.00,0.00,304.00,24.05,304.00,304.00,304.00,76.16,0.00,304.00,0.00,100.0,0.00,0.0,0.0,50.00,50.00,24.49,0.00,100.0,10

In [6]:
df[['fpd', 'safra', 'rec_qtd_geral']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2633900 entries, 0 to 2633899
Data columns (total 3 columns):
 #   Column         Dtype         
---  ------         -----         
 0   fpd            bool          
 1   safra          datetime64[us]
 2   rec_qtd_geral  int64         
dtypes: bool(1), datetime64[us](1), int64(1)
memory usage: 42.7 MB


In [7]:
print(f"Linhas: {len(df):,}")
print(f"Colunas: {df.shape[1]}")
print(f"Safras: {df['safra'].nunique()}")
print(f"Produtos: {df['prod'].nunique()}")
print(f"FPD médio: {df['fpd'].mean()*100:.2f}%")

df["fpd"] = df["fpd"].astype(int)
df["safra"] = pd.to_datetime(df["safra"])

Linhas: 2,633,900
Colunas: 207
Safras: 6
Produtos: 1
FPD médio: 21.23%


### Checklist A: vazamento temporal (datas ≥ safra)

In [8]:
date_cols = [c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c]) and c != "ingestion_ts"]
df[date_cols].head(3)

,safra,rec_dat_primeira,rec_dat_ultima,atr_dat_ultima_ref,cad_datadenascimento,cad_var_12,cad_var_13
0,2024-10-01,2023-10-22,2024-09-28,NaT,1998-11-23,NaT,NaT
1,2024-10-01,2023-10-30,2024-09-30,NaT,2003-12-27,NaT,NaT
2,2024-10-01,2023-10-09,2024-09-19,NaT,1974-12-13,2007-10-01,NaT


In [9]:
results = []
for c in date_cols:
    pct_future = (pd.to_datetime(df[c], errors="coerce") >= df["safra"]).mean()
    results.append({
        "feature": c,
        "pct_date_ge_safra": pct_future,
        "pct_missing": df[c].isna().mean()
    })

df_dates_audit = (
    pd.DataFrame(results)
      .sort_values("pct_date_ge_safra", ascending=False)
)

df_dates_audit.head(20)

,feature,pct_date_ge_safra,pct_missing
0,safra,1.000000,0.000000
6,cad_var_13,0.021091,0.849987
1,rec_dat_primeira,0.000000,0.281699
2,rec_dat_ultima,0.000000,0.281699
3,atr_dat_ultima_ref,0.000000,0.681503
4,cad_datadenascimento,0.000000,0.002270
5,cad_var_12,0.000000,0.353985


In [10]:
df['cad_var_13'].head(10)

0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
5   NaT
6   NaT
7   NaT
8   NaT
9   NaT
Name: cad_var_13, dtype: datetime64[us]

Aparentemente temos vazamento de features posteriores a safra somente em cad_var_13 --> aproximadamente 2.15% --> podemos descartar essa variável para evitar ruído;

### Checklist B: auditoria semântica por domínio

In [11]:
df_features = pd.DataFrame({
    "feature": df.columns
})

In [12]:
df_features["domain"] = df_features["feature"].apply(feature_domain)

In [13]:
df_features["domain"].value_counts()

domain
tel          69
rec          39
pag          36
cad          28
atr          24
bur           3
flag          1
fpd           1
prod          1
safra         1
num           1
run           1
ingestion     1
ano           1
Name: count, dtype: int64

In [14]:
df_features["dtype"] = df.dtypes.values

In [15]:
df_features.head(20)

,feature,domain,dtype
0,num_cpf,num,object
1,safra,safra,datetime64[us]
2,prod,prod,object
3,fpd,fpd,int64
4,flag_instalacao,flag,bool
5,rec_qtd_l30d,rec,int64
6,rec_qtd_l60d,rec,int64
7,rec_qtd_l90d,rec,int64
8,rec_qtd_geral,rec,int64
9,rec_vlr_total_l30d,rec,float64


### Checklist C1: AUC univariada (detector estatístico)

In [16]:
exclude = {
    "num_cpf","safra","prod","fpd",
    "run_id","ingestion_ts","ano_mes"
}

rows = []
y = df["fpd"].values

In [17]:
for c in df.columns:
    if c in exclude:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        x = df[c].fillna(df[c].median())
        try:
            auc = roc_auc_score(y, x)
            rows.append({
                "feature": c,
                "auc_univariada": auc,
                "pct_missing": df[c].isna().mean()
            })
        except Exception as e:
            print(e)
            pass

In [18]:
df_auc = (
    pd.DataFrame(rows)
      .sort_values("auc_univariada", ascending=False)
)

df_auc.head(30)

,feature,auc_univariada,pct_missing
114,tel_var_31,0.561484,0.503476
72,pag_flag_sem_pagamento_l60d,0.551689,0.000000
71,pag_flag_sem_pagamento_l30d,0.550650,0.000000
73,pag_flag_sem_pagamento_l90d,0.550337,0.000000
111,tel_var_28,0.549863,0.503476
21,rec_qtd_canais_distintos,0.549030,0.000000
133,tel_var_50,0.542311,0.503476
129,tel_var_46,0.542248,0.503476
131,tel_var_48,0.539259,0.503476
30,rec_vlr_coef_var_l90d,0.532910,0.527925


In [19]:
blacklist = df_auc[df_auc["auc_univariada"] > 0.80]
whitelist = df_auc[df_auc["auc_univariada"] <= 0.80]

print("BLACKLIST (suspeita forte de vazamento):")
display(blacklist.head(20))

print("WHITELIST (aprovadas para baseline):")
display(whitelist.head(20))


BLACKLIST (suspeita forte de vazamento):


,feature,auc_univariada,pct_missing


WHITELIST (aprovadas para baseline):


,feature,auc_univariada,pct_missing
114,tel_var_31,0.561484,0.503476
72,pag_flag_sem_pagamento_l60d,0.551689,0.000000
71,pag_flag_sem_pagamento_l30d,0.550650,0.000000
73,pag_flag_sem_pagamento_l90d,0.550337,0.000000
111,tel_var_28,0.549863,0.503476
21,rec_qtd_canais_distintos,0.549030,0.000000
133,tel_var_50,0.542311,0.503476
129,tel_var_46,0.542248,0.503476
131,tel_var_48,0.539259,0.503476
30,rec_vlr_coef_var_l90d,0.532910,0.527925
